# Savvy · AI Lead Triage — reproduce the experiment

This notebook runs the full workflow end to end on the fictional Savvy lead dataset:

1. **Setup**: gets the code and installs the pinned dependencies
2. **API key**: asks for an OpenAI key (hidden input, never printed or saved)
3. **Load**: downloads the CSV from the web, so there's nothing to upload by hand
4. **Validate & normalize**: rules in code
5. **Duplicates**: linked, never merged
6. **AI classification**: `gpt-4o-mini`, with guardrails enforced in code
7. **Routing, cost and export**: plus the automatic PASS / FAIL test cases

**How to run:** run each cell from top to bottom (`Shift + Enter`). It works locally inside the repo or on Google Colab.
A full run makes 19 API calls and costs well under one cent.

## 1. Setup

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/carlicode/Savvy_Practical_Assesment"

# Inside the repo already? Use it. Otherwise (e.g. Colab) clone it.
if not Path("pipeline").exists():
    if not Path("Savvy_Practical_Assesment").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("Savvy_Practical_Assesment")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", "requirements.txt"], check=True)
sys.path.insert(0, str(Path.cwd()))
print("Ready. Working directory:", Path.cwd().name)

Ready. Working directory: Savvy_Practical_Assesment


## 2. API key

The key is read from a local `.env` file if one exists. Otherwise you'll get a hidden prompt.
It's kept only in memory for this session: it isn't printed, saved or written into the notebook.

In [2]:
import getpass
from dotenv import load_dotenv

load_dotenv(".env")
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste the OpenAI API key (hidden): ").strip()

print("API key loaded:", "yes" if os.getenv("OPENAI_API_KEY") else "no")

API key loaded: yes


## 3. Load the leads from the web

`DATA_URL` points to the public CSV. To use a copy hosted elsewhere (for example an S3 object URL), change this one line.

In [3]:
import urllib.request
from IPython.display import HTML, display

from pipeline.data import load_leads, validate_leads
from pipeline.duplicates import detect_duplicates, print_duplicates
from pipeline.ai import classify_leads_with_ai
from pipeline.output import export_results, OUTPUT_PATH

DATA_URL = "https://raw.githubusercontent.com/carlicode/Savvy_Practical_Assesment/main/data/Savvy_Mock_Lead_Dataset.csv"
LOCAL_CSV = Path("data") / "downloaded_leads.csv"

LOCAL_CSV.parent.mkdir(exist_ok=True)
urllib.request.urlretrieve(DATA_URL, LOCAL_CSV)
leads = load_leads(LOCAL_CSV)
print(f"Downloaded {len(leads)} leads from {DATA_URL.split('/')[2]}")


def show(rows, columns):
    """Render a list of dicts as a small HTML table."""
    head = "".join(f"<th style='text-align:left'>{c}</th>" for c in columns)
    body = "".join(
        "<tr>" + "".join(f"<td style='text-align:left'>{row.get(c, '')}</td>" for c in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))


show(leads[:5], ["Lead_ID", "Lead_Source", "Lead_Type", "Property_Location", "Budget_USD", "Purchase_or_Action_Timeline"])

Downloaded 20 leads from raw.githubusercontent.com


Lead_ID,Lead_Source,Lead_Type,Property_Location,Budget_USD,Purchase_or_Action_Timeline
SAV-1001,Facebook Ads,STR Buyer,"Asheville, NC",750000.0,30 days
SAV-1002,Website,STR Seller,"Hendersonville, NC",625000.0,60 days
SAV-1003,Referral,STR Buyer,"Asheville, NC",900000.0,ASAP
SAV-1004,Facebook Ads,STR Buyer,"Black Mountain, NC",500000.0,3-6 months
SAV-1005,Website,STR Buyer,"Weaverville, NC",550000.0,Unknown


## 4. Validate & normalize (code, no AI)

Email lowercased, phone as `+1XXXXXXXXXX`, budget as a number, timeline in days. Missing or doubtful data gets a flag.
A missing budget is normal for Property Management, so it isn't flagged there.

In [4]:
validated = validate_leads(leads)
show(validated, ["Lead_ID", "Email_Normalized", "Phone_Normalized", "Budget_USD_Normalized", "Timeline_Days", "Data_Flags"])

Lead_ID,Email_Normalized,Phone_Normalized,Budget_USD_Normalized,Timeline_Days,Data_Flags
SAV-1001,sarah.mitchell@example.com,+12025550101,750000.0,30,
SAV-1002,daniel.brooks@example.com,+12025550102,625000.0,60,
SAV-1003,emily.carter@example.com,+12025550103,900000.0,0,
SAV-1004,,+12025550104,500000.0,90,email_missing
SAV-1005,olivia.reed@example.com,,550000.0,,phone_missing;timeline_missing
SAV-1006,noah.bennett@example.com,+12025550106,450000.0,90,
SAV-1007,ava.foster@example.com,+12025550107,710000.0,30,
SAV-1008,liam.cooper@example.com,+12025550108,,30,
SAV-1009,sophia.hayes@example.com,+12025550109,800000.0,30,
SAV-1010,james.perry@example.com,+12025550110,650000.0,0,


## 5. Duplicates (code, no AI)

Same email or phone means a confirmed duplicate. It's **linked** to the first record with `duplicate_of`, never merged or deleted.

In [5]:
validated = detect_duplicates(validated)
print_duplicates(validated)


Duplicates found: 1
Lead_ID    duplicate_of   Contact
----------------------------------------------------------------------
SAV-1011   SAV-1001       sarah.mitchell@example.com / +12025550101

Possible duplicates (same name + location, different contact): 0
None


## 6. AI classification

For each lead that isn't a duplicate, `gpt-4o-mini` returns intent, priority, confidence, a next action and a draft follow-up, validated against a schema.
The model never receives a name, email, phone, Lead_ID or `Data_Notes`. The human-review rules are applied in code afterwards.

In [6]:
validated = classify_leads_with_ai(validated)


AI classification
----------------------------------------------------------------------------------------------------


SAV-1001: Buyer | High | 0.90 | review=false | tokens=448/139 | The lead has a clear intent to purchase a short-term rental investment within 30 days and has a budget of $750,000. They also express a desire to move quickly, indicating urgency.
Next: Schedule a property tour in Asheville.
Draft: Hi there! Thank you for your interest in short-term rental investments in Asheville. Given your timeline, I'd love to schedule a property tour for you this week. Please let me know your availability.


SAV-1002: Seller | Medium | 0.80 | review=false | tokens=448/140 | The lead is considering selling an established vacation rental and has a clear timeline of 60 days. They are seeking a valuation and net proceeds estimate, indicating serious intent.
Next: Provide a valuation and net proceeds estimate.
Draft: Hi there, thank you for reaching out! I would love to assist you with the valuation of your vacation rental in Hendersonville. Could you please provide any recent updates or improvements made to the property that might affect its value?


SAV-1003: Buyer | High | 0.90 | review=false | tokens=450/129 | The lead has financing ready and is looking to tour properties this week, indicating a strong intent to purchase. The timeline is ASAP, which elevates the urgency.
Next: Schedule a call today to discuss suitable properties and arrange tours.
Draft: Hi there! I'm excited to help you find suitable properties in Asheville. Let's schedule a call today to discuss your preferences and arrange some tours for this week.


SAV-1004: Buyer | Medium | 0.70 | review=false | tokens=449/138 | The lead has a clear intent to purchase a property within 3-6 months and has a specified budget. However, they have not provided an email address, which limits communication options.
Next: Request the lead's email address for further communication.
Draft: Hi there! Thanks for your interest in properties in Black Mountain, NC. Could you please share your email address so I can send you more detailed information? Looking forward to assisting you!


SAV-1005: Buyer | Medium | 0.60 | review=true | tokens=446/154 | The lead has expressed interest in investment properties but has not provided a clear timeline for their purchase. The budget indicates serious intent, but the unknown timeline affects urgency.
Next: Reach out to the lead to clarify their timeline and discuss potential properties.
Draft: Hi there! Thank you for your interest in investment properties in Weaverville, NC. I noticed you didn't provide a phone number; could you share your preferred contact number? Additionally, it would be helpful to know your timeline for purchasing so we can assist you better.


... (10 outputs left out)

SAV-1017: Seller | Medium | 0.70 | review=false | tokens=448/146 | The lead is a seller looking for a market value estimate with a clear 90-day timeline. They are not ready to list yet, indicating a medium priority.
Next: Provide a market value estimate based on current data.
Draft: Hi there, thank you for reaching out! I can provide you with an estimate of your property's current market value in Weaverville, NC. Please let me know if there are any specific details about your property that you would like me to consider.


SAV-1018: Buyer | High | 0.90 | review=false | tokens=449/154 | The lead is an experienced investor with a clear intent to purchase a higher-end property within 30 days, indicating urgency. They are also looking to speak with a specialist, which suggests they are serious about their investment.
Next: Schedule a call with a specialist to discuss their needs and available properties.
Draft: Hi there, thank you for reaching out! I’d love to connect you with one of our specialists to discuss your interest in higher-end properties in Asheville. Please let me know your availability for a call this week.


SAV-1019: Unclear | Low | 0.40 | review=true | tokens=452/140 | The lead shows interest but lacks specific details about their intent, timeline, and budget. This makes it difficult to assess urgency or clear intent.
Next: Send an introductory email with general information about available properties.
Draft: Hi there! Thank you for your interest in our properties in Asheville, NC. I would love to provide you with more details. Could you please share what specific information you are looking for or any particular features you have in mind?


SAV-1020: Buyer | High | 0.90 | review=false | tokens=446/144 | The lead has proof of funds and is looking to schedule a showing within 48 hours, indicating urgency. The timeline is ASAP, which qualifies for high priority.
Next: Schedule a showing within the next 48 hours.
Draft: Hi there! Thank you for your interest in properties in Arden, NC. I see that you're ready to schedule a showing within the next 48 hours. Please let me know your availability so we can arrange a time that works for you.
Saved to output/ai_classification_2026-11-05_21-06.txt


## 7. Routing, cost, export and test cases

Routing is decided by rules, not by the AI. The processed queue is written to `output/processed_leads.csv`, then the four test cases from the brief are checked.

In [7]:
validated = export_results(validated)


Routing and cost
----------------------------------------------------------------------------------------------------
SAV-1001   Buyer team             suggested=Buyer team             $0.000151
SAV-1003   Buyer team             suggested=Buyer team             $0.000145
SAV-1009   Buyer team             suggested=Buyer team             $0.000155
SAV-1010   Human review           suggested=Buyer team             $0.000156
SAV-1011   Human review           suggested=Buyer team             $0.000000
SAV-1013   Seller team            suggested=Seller team            $0.000147
SAV-1016   Buyer team             suggested=Buyer team             $0.000161
SAV-1018   Buyer team             suggested=Buyer team             $0.000160
SAV-1020   Buyer team             suggested=Buyer team             $0.000153
SAV-1002   Seller team            suggested=Seller team            $0.000151
SAV-1004   Buyer team             suggested=Buyer team             $0.000150
SAV-1005   Human review           

### The processed lead queue

In [8]:
import csv

with open(OUTPUT_PATH, newline="", encoding="utf-8") as handle:
    processed = list(csv.DictReader(handle))

show(processed, ["Lead_ID", "intent", "priority", "confidence", "needs_human_review", "route", "suggested_team", "next_action", "cost_usd"])
print(f"{len(processed)} leads · total AI cost ${sum(float(r['cost_usd']) for r in processed):.6f}")

Lead_ID,intent,priority,confidence,needs_human_review,route,suggested_team,next_action,cost_usd
SAV-1001,Buyer,High,0.90,false,Buyer team,Buyer team,Schedule a property tour in Asheville.,0.000151
SAV-1003,Buyer,High,0.90,false,Buyer team,Buyer team,Schedule a call today to discuss suitable properties and arrange tours.,0.000145
SAV-1009,Buyer,High,0.90,false,Buyer team,Buyer team,Schedule a property tour or call to discuss options.,0.000155
SAV-1010,Buyer,High,0.70,true,Human review,Buyer team,"Provide information on STR regulations and investment risks in Asheville, NC.",0.000156
SAV-1011,Buyer,High,0.90,false,Human review,Buyer team,"No follow-up, duplicate of SAV-1001. Confirm in CRM.",0.000000
SAV-1013,Seller,High,0.90,false,Seller team,Seller team,Schedule a call or meeting to discuss the sale details.,0.000147
SAV-1016,Buyer,High,0.90,false,Buyer team,Buyer team,"Provide information about short-term rental regulations in Hendersonville, NC and schedule a property tour.",0.000161
SAV-1018,Buyer,High,0.90,false,Buyer team,Buyer team,Schedule a call with a specialist to discuss their needs and available properties.,0.000160
SAV-1020,Buyer,High,0.90,false,Buyer team,Buyer team,Schedule a showing within the next 48 hours.,0.000153
SAV-1002,Seller,Medium,0.80,false,Seller team,Seller team,Provide a valuation and net proceeds estimate.,0.000151


20 leads · total AI cost $0.002900


## Done

- `output/processed_leads.csv`: the full queue with every required field
- `output/ai_classification_<date>.txt`: the log of this run

**Note:** AI outputs can vary slightly between runs, even at temperature 0. Every run is logged so you can compare them.